In [ ]:
# Install required packages for Colab
!pip install ecdsa
!pip install pycryptodome

import hashlib
import ecdsa
import binascii
from datetime import datetime
from Crypto.Hash import RIPEMD160

# Base58 alphabet
BASE58_ALPHABET = '123456789ABCDEFGHJKLMNPQRSTUVWXYZabcdefghijkmnopqrstuvwxyz'

def base58check_encode(version, payload):
    """Encode version + payload into Base58Check with checksum."""
    data = version + payload
    checksum = hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]
    full_data = data + checksum
    num = int.from_bytes(full_data, 'big')
    result = ''
    while num > 0:
        num, remainder = divmod(num, 58)
        result = BASE58_ALPHABET[remainder] + result
    for byte in full_data:
        if byte != 0:
            break
        result = '1' + result
    return result

def compute_checksum(version, payload):
    """Compute Base58Check checksum for version + payload."""
    data = version + payload
    return hashlib.sha256(hashlib.sha256(data).digest()).digest()[:4]

def generate_p2pkh_hash160(private_key):
    """Generate Hash160 from private key using pycryptodome for RIPEMD160."""
    sk = ecdsa.SigningKey.from_string(private_key, curve=ecdsa.SECP256k1)
    vk = sk.verifying_key
    public_key = b'\x04' + vk.to_string()  # Uncompressed
    sha256_hash = hashlib.sha256(public_key).digest()
    ripemd160_hash = RIPEMD160.new()
    ripemd160_hash.update(sha256_hash)
    return ripemd160_hash.digest()

def generate_seed_from_timestamp(timestamp):
    """Generate private key from timestamp-based seed."""
    seed = f"mtgox-{timestamp:.9f}"
    return hashlib.sha256(seed.encode()).digest()

def main():
    # Target address details
    target_address = "1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF"
    target_prefix = "1Feex"
    target_hash160 = binascii.unhexlify("a0b0d60e5991578ed37cbda2b17d8b2ce23ab295")
    version = b'\x00'  # Fixed P2PKH version
    target_checksum = binascii.unhexlify("8cb6938f")  # Known checksum
    print(f"Target Address: {target_address}")
    print(f"Target Hash160: {target_hash160.hex()}")
    print(f"Target Checksum: {target_checksum.hex()}")

    # Simulate 2011-era timestamps (March 1, 2011 ± 1 day)
    start_time = 1298937600  # March 1, 2011, 00:00:00 UTC
    end_time = start_time + 24 * 3600  # 1 day later
    step = 0.001  # Millisecond steps
    max_attempts = 50000  # Colab-friendly

    print(f"\nFiltering by checksum {target_checksum.hex()} for {target_address}...")
    matches = []
    checksum_matches = 0
    attempt = 0

    for ts in range(int(start_time * 1000), int(end_time * 1000), int(step * 1000)):
        if attempt >= max_attempts:
            break
        timestamp = ts / 1000.0
        # Generate private key
        private_key = generate_seed_from_timestamp(timestamp)
        # Generate Hash160
        hash160 = generate_p2pkh_hash160(private_key)
        # Filter by checksum
        checksum = compute_checksum(version, hash160)
        if checksum == target_checksum:
            checksum_matches += 1
            # Generate full address
            address = base58check_encode(version, hash160)
            # Check for prefix or exact match
            if address.startswith(target_prefix):
                matches.append((address, hash160.hex(), timestamp))
                print(f"Prefix Match! Address: {address}, Hash160: {hash160.hex()}, Timestamp: {timestamp}")
            if hash160 == target_hash160:
                print(f"*** Exact Match for 1Feex! Seed: mtgox-{timestamp:.9f} ***")
                break
            print(f"Checksum Match #{checksum_matches}: Address: {address}, Timestamp: {timestamp}")
        attempt += 1
        if attempt % 10000 == 0:
            print(f"Processed {attempt} attempts...")

    # Report results
    print(f"\nTotal attempts: {attempt}")
    print(f"Checksum matches: {checksum_matches}")
    print(f"Prefix '{target_prefix}' matches: {len(matches)}")
    for i, (addr, h160, ts) in enumerate(matches, 1):
        print(f"Prefix Match {i}: Address: {addr}, Hash160: {h160}, Timestamp: {ts}")
    if checksum_matches > 0:
        print(f"Checksum filter hit rate: {checksum_matches/attempt:.6%}")
    else:
        print("No checksum matches found.")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 31.9 MB/s eta 0:00:00
Target Address: 1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF
Target Hash160: a0b0d60e5991578ed37cbda2b17d8b2ce23ab295
Target Checksum: 8cb6938f

Filtering by checksum 8cb6938f for 1FeexV6bAHb8ybZjqQMjJrcCrHGW9sb6uF...
Processed 10000 attempts...
Processed 20000 attempts...
Processed 30000 attempts...
Processed 40000 attempts...
Processed 50000 attempts...

Total attempts: 50000
Checksum matches: 0
Prefix '1Feex' matches: 0
No checksum matches found.


In [ ]:
#!/usr/bin/env python3
# Enhanced Tinkering Script for Date-Based and High-Entropy Keys (May 17, 2014, Clean Syntax)

import hashlib
import base58
import random
import ecdsa
from Crypto.Hash import RIPEMD160

def generate_address(private_key_int):
    try:
        private_key_hex = format(private_key_int, '064x')
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()
        public_key = b'\x04' + vk.to_string()
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160_hasher = RIPEMD160.new()
        ripemd160_hasher.update(sha256_hash)
        hash160 = ripemd160_hasher.digest()
        versioned_hash = b'\x00' + hash160
        checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
        address_bytes = versioned_hash + checksum
        address = base58.b58encode(address_bytes).decode('utf-8')
        return address, hash160
    except:
        return None, None

def hamming_distance(hash1, hash2):
    try:
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except:
        return 160

def tinker_date_key(target_hash160, target_addr):
    results = []
    with open('tinkering_results.txt', 'a') as f:
        # Date range: May 15-19, 2014, plus broader checks
        dates = []
        for day in range(15, 20):  # May 15-19, 2014
            dates.append((2014, 5, day))
        for _ in range(20):  # Additional 2009-2014 dates
            year = random.randint(2009, 2014)
            month = random.randint(1, 12)
            day = random.randint(1, 31)
            dates.append((year, month, day))
        for _ in range(10):  # Additional 1950-2025 dates
            year = random.randint(1950, 2025)
            month = random.randint(1, 12)
            day = random.randint(1, 31)
            dates.append((year, month, day))

        formats = [
            "{year}{month:02d}{day:02d}",
            "{day:02d}{month:02d}{year}",
            "{month:02d}{day:02d}{year}",
            "{year}{day:02d}{month:02d}",
            "{year}-{month:02d}-{day:02d}",
            "{day:02d}-{month:02d}-{year}",
            "{year}{month:02d}",
            "{month:02d}{day:02d}",
            "{day:02d}{month:02d}",
            "{year[2:4]}{month:02d}{day:02d}",
            "{day:02d}May{year}",
            "{year}{month:02d}{day:02d}120000"
        ]
        suffixes = ['0000', 'ffff', '5555', '1111', '2014', '0517', '8888', '9999', '7777', '']
        prefixes = [''] + [''.join(random.choice('0123456789abcdef') for _ in range(48)) for _ in range(20)]
        passphrases = ['20140517', 'may172014', '17may2014', '2014-05-17', 'wallet20140517']

        # Test date-based keys
        for year, month, day in dates:
            for prefix in prefixes:
                for format_str in formats:
                    for suffix in suffixes:
                        try:
                            key_str = prefix + format_str.format(year=str(year), month=month, day=day) + suffix
                            key_str = key_str[:64].ljust(64, '0')
                            key_int = int(key_str, 16)
                            address, hash160 = generate_address(key_int)
                            if address == target_addr:
                                print(f"EXACT MATCH for {target_addr}: {format(key_int, '064x')}")
                                f.write(f"EXACT MATCH,{target_addr},{format(key_int, '064x')},0\n")
                                return key_int, 0
                            hamming = hamming_distance(hash160, target_hash160)
                            if hamming < 50:
                                results.append((key_int, hamming))
                                f.write(f"Close match,{target_addr},{format(key_int, '064x')},{hamming}\n")
                                print(f"Close match for {target_addr}: {format(key_int, '064x')} (Hamming: {hamming})")
                        except:
                            continue

        # Test passphrase-based keys
        for passphrase in passphrases:
            for i in range(100):  # Variations with counter
                try:
                    key_str = hashlib.sha256((passphrase + str(i)).encode()).hexdigest()
                    key_int = int(key_str, 16)
                    address, hash160 = generate_address(key_int)
                    if address == target_addr:
                        print(f"EXACT MATCH for {target_addr}: {format(key_int, '064x')}")
                        f.write(f"EXACT MATCH,{target_addr},{format(key_int, '064x')},0\n")
                        return key_int, 0
                    hamming = hamming_distance(hash160, target_hash160)
                    if hamming < 50:
                        results.append((key_int, hamming))
                        f.write(f"Close match,{target_addr},{format(key_int, '064x')},{hamming}\n")
                        print(f"Close match for {target_addr}: {format(key_int, '064x')} (Hamming: {hamming})")
                except:
                    continue

    return results

# Target addresses
addresses = [
    '1BFYFjbvGUUGxw3t3TbmH6xyKySw4TT2VN',
    '12Bb5bVwEqyMY66hEXG1Go5Hph39jRS33L'
]

# Initialize results file
with open('tinkering_results.txt', 'w') as f:
    f.write("Type,Address,Private Key,Hamming Distance\n")

# Run tinkering for both addresses
for addr in addresses:
    print(f"\nTesting variations for {addr}")
    decoded = base58.b58decode(addr)
    target_hash160 = decoded[1:-4]
    results = tinker_date_key(target_hash160, addr)
    if results:
        print(f"Found {len(results)} close matches for {addr}:")
        for key_int, hamming in results:
            print(f"  Key: {format(key_int, '064x')}, Hamming: {hamming}")
    else:
        print(f"No closer matches found for {addr}")


Testing variations for 1BFYFjbvGUUGxw3t3TbmH6xyKySw4TT2VN
No closer matches found for 1BFYFjbvGUUGxw3t3TbmH6xyKySw4TT2VN

Testing variations for 12Bb5bVwEqyMY66hEXG1Go5Hph39jRS33L
No closer matches found for 12Bb5bVwEqyMY66hEXG1Go5Hph39jRS33L


In [ ]:
#!/usr/bin/env python3
# Bitcoin Key Counter Analyzer - Sequential Keys Starting from k=1

# Install required packages
!pip install -q base58 colorama tqdm ecdsa pycryptodome

import hashlib
import base58
import os
import time
from colorama import Fore, init
from tqdm.notebook import tqdm
from google.colab import files
import signal
import sys
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import pickle
import binascii
import ecdsa
from Crypto.Hash import RIPEMD160

# Initialize colorama
init(autoreset=True)

# SECP256k1 curve order (n)
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Global variables for child processes
TARGET_ADDRESSES = []
TARGET_HASH160S = {}

def generate_address(private_key_int):
    try:
        # Validate key range
        if not (0 < private_key_int < CURVE_ORDER):
            with open('error_log.txt', 'a') as f:
                f.write(f"Invalid key in generate_address: {private_key_int} (out of range 1 to {CURVE_ORDER-1})\n")
            return None, None
        private_key_hex = format(private_key_int, '064x')
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()
        public_key = b'\x04' + vk.to_string()
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160_hasher = RIPEMD160.new()
        ripemd160_hasher.update(sha256_hash)
        hash160 = ripemd160_hasher.digest()
        versioned_hash = b'\x00' + hash160
        checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
        address_bytes = versioned_hash + checksum
        address = base58.b58encode(address_bytes).decode('utf-8')
        return address, hash160
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error in generate_address for key {private_key_int}: {str(e)}\n")
        return None, None

def hamming_distance(hash1, hash2):
    try:
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except:
        return 160

def process_counter_batch(batch_data):
    start_key, batch_size = batch_data
    results = []
    try:
        with open('close_keys.txt', 'a') as close_keys_file:
            for k in range(start_key, start_key + batch_size):
                key_int = k
                address, hash160 = generate_address(key_int)
                if not address or not hash160:
                    continue
                for target_addr, target_hash160 in TARGET_HASH160S.items():
                    if random.random() < 0.1:  # Uniform 10% sampling
                        hamming = hamming_distance(hash160, target_hash160)
                        if hamming < 50:
                            result = {
                                'target': target_addr,
                                'key_int': key_int,
                                'private_key': format(key_int, '064x'),
                                'hamming': hamming,
                                'exact_match': False,
                                'entropy': 0.0  # Counter-based keys have low entropy
                            }
                            results.append(result)
                            if hamming <= 45:
                                close_keys_file.write(f"{target_addr},{result['private_key']},{hamming},0.00\n")
                    if address == target_addr:
                        result = {
                            'target': target_addr,
                            'key_int': key_int,
                            'private_key': format(key_int, '064x'),
                            'hamming': 0,
                            'exact_match': True,
                            'entropy': 0.0
                        }
                        results.append(result)
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error in process_counter_batch for start_key {start_key}: {str(e)}\n")
    return results

class FastCounterAnalyzer:
    def __init__(self):
        self.target_addresses = []
        self.results = {}
        self.best_matches = {}
        self.checkpoint_file = 'counter_analyzer.pkl'
        self.cores = multiprocessing.cpu_count()
        self.running = True
        self.report_file = 'counter_report.txt'
        self.silent_mode = False
        signal.signal(signal.SIGINT, self.handle_interrupt)

    def handle_interrupt(self, sig, frame):
        print(f"\n{Fore.YELLOW}Interrupt received, saving progress...")
        self.running = False
        self.save_checkpoint()
        time.sleep(1)
        sys.exit(0)

    def load_addresses(self):
        print(f"{Fore.CYAN}Processing addresses...")
        try:
            with open("addresses.txt", 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]
        except:
            try:
                with open("addresses (1).txt", 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]
            except:
                print(f"{Fore.CYAN}Upload a text file with Bitcoin addresses (one per line)")
                uploaded = files.upload()
                if not uploaded:
                    print(f"{Fore.RED}No file uploaded. Exiting.")
                    return False
                filename = list(uploaded.keys())[0]
                with open(filename, 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]

        valid_addresses = []
        global TARGET_ADDRESSES, TARGET_HASH160S
        TARGET_ADDRESSES = []
        TARGET_HASH160S = {}
        for addr in addresses:
            if not addr.startswith('1'):
                continue
            try:
                decoded = base58.b58decode(addr)
                if len(decoded) != 25:
                    continue
                hash160 = decoded[1:-4]
                valid_addresses.append(addr)
                TARGET_ADDRESSES.append(addr)
                TARGET_HASH160S[addr] = hash160
                self.results[addr] = []
            except:
                continue
        if not valid_addresses:
            print(f"{Fore.RED}No valid P2PKH addresses found.")
            return False
        self.target_addresses = valid_addresses
        print(f"{Fore.GREEN}Loaded {len(valid_addresses)} valid addresses.")
        return True

    def save_checkpoint(self):
        checkpoint_data = {
            'targets': self.target_addresses,
            'results': self.results,
            'best_matches': self.best_matches
        }
        try:
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
            if not self.silent_mode:
                print(f"{Fore.GREEN}Progress saved.")
            return True
        except Exception as e:
            print(f"{Fore.RED}Error saving checkpoint: {str(e)}")
            return False

    def load_checkpoint(self):
        if not os.path.exists(self.checkpoint_file):
            return False
        try:
            with open(self.checkpoint_file, 'rb') as f:
                data = pickle.load(f)
            self.target_addresses = data['targets']
            self.results = data['results']
            self.best_matches = data.get('best_matches', {})
            global TARGET_ADDRESSES, TARGET_HASH160S
            TARGET_ADDRESSES = self.target_addresses.copy()
            TARGET_HASH160S = {}
            for addr in self.target_addresses:
                try:
                    decoded = base58.b58decode(addr)
                    hash160 = decoded[1:-4]
                    TARGET_HASH160S[addr] = hash160
                except:
                    pass
            print(f"{Fore.GREEN}Loaded checkpoint with {len(self.target_addresses)} addresses.")
            return True
        except Exception as e:
            print(f"{Fore.RED}Error loading checkpoint: {str(e)}")
            return False

    def run(self, iterations=5000, batch_factor=2.0):
        if not self.load_checkpoint():
            if not self.load_addresses():
                return False

        print(f"{Fore.GREEN}Starting counter-based key analysis for {len(self.target_addresses)} addresses")
        print(f"{Fore.GREEN}Using {self.cores} CPU cores for maximum speed")
        print(f"{Fore.GREEN}Press Ctrl+C to pause analysis and save progress")
        print(f"{Fore.GREEN}Report will be saved to {self.report_file}")

        start_time = time.time()
        current_key = 1  # Start at k=1

        try:
            with open(self.report_file, 'w') as f:
                f.write(f"=== BITCOIN COUNTER ANALYSIS REPORT ===\n")
                f.write(f"Started at: {time.ctime()}\n")
                f.write(f"Analyzing {len(self.target_addresses)} addresses\n\n")

            with open('close_keys.txt', 'w') as f:
                f.write("Address,Private Key,Hamming Distance,Entropy Level\n")
            with open('error_log.txt', 'w') as f:
                f.write("Error Log\n")

            for iteration in range(iterations):
                if not self.running:
                    break

                exact_matches = sum(1 for addr in self.target_addresses
                                   if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
                if exact_matches == len(self.target_addresses):
                    print(f"{Fore.GREEN}All private keys recovered successfully!")
                    break

                batch_size = int(2500 * batch_factor)  # Keys per batch
                batches = [(current_key + i * batch_size, batch_size) for i in range(self.cores)]
                current_key += batch_size * self.cores

                if iteration % 50 == 0 or iteration == 0:
                    print(f"\n{Fore.CYAN}Iteration {iteration+1}/{iterations} - Testing keys {batches[0][0]} to {current_key-1}")
                else:
                    self.silent_mode = True

                with ProcessPoolExecutor(max_workers=self.cores) as executor:
                    futures = [executor.submit(process_counter_batch, batch) for batch in batches]
                    for future in futures:
                        try:
                            batch_results = future.result()
                            for result in batch_results:
                                if not isinstance(result, dict):
                                    with open('error_log.txt', 'a') as f:
                                        f.write(f"Invalid result type in batch_results: {type(result)}\n")
                                    continue
                                addr = result['target']
                                self.results[addr].append(result)
                                if addr not in self.best_matches or result['hamming'] < self.best_matches[addr]['hamming']:
                                    self.best_matches[addr] = result
                                    if not self.silent_mode or result['hamming'] < 38:
                                        if result['exact_match']:
                                            print(f"{Fore.GREEN}EXACT MATCH FOUND for {addr[:8]}...{addr[-4:]}")
                                        elif result['hamming'] < 38:
                                            print(f"{Fore.YELLOW}BRUTE-FORCE RANGE for {addr[:8]}...{addr[-4:]} (Hamming: {result['hamming']})")
                        except Exception as e:
                            with open('error_log.txt', 'a') as f:
                                f.write(f"Error processing future result: {str(e)}\n")

                if iteration % 50 == 0 or iteration == iterations-1:
                    self.silent_mode = False
                    self.display_progress()
                    self.save_checkpoint()
                    self.update_report_file()
                else:
                    self.save_checkpoint()

            self.silent_mode = False
            self.save_checkpoint()
            self.generate_final_report()

            elapsed = time.time() - start_time
            print(f"{Fore.GREEN}Analysis completed in {elapsed:.2f} seconds")
            print(f"{Fore.GREEN}Final report saved to {self.report_file}")
            print(f"{Fore.GREEN}Close keys saved to close_keys.txt")

            try:
                files.download(self.report_file)
                print(f"{Fore.GREEN}Report downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download report automatically. Please download it manually.")

            try:
                files.download('best_matches.csv')
                print(f"{Fore.GREEN}Best matches CSV downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download CSV automatically. Please download it manually.")

            try:
                files.download('close_keys.txt')
                print(f"{Fore.GREEN}Close keys downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download close_keys.txt automatically. Please download it manually.")

            try:
                files.download('error_log.txt')
                print(f"{Fore.GREEN}Error log downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download error_log.txt automatically. Please download it manually.")

            return True

        except KeyboardInterrupt:
            print(f"\n{Fore.YELLOW}Analysis interrupted. Saving progress...")
            self.save_checkpoint()
            self.update_report_file()
            return False
        except Exception as e:
            print(f"{Fore.RED}Unexpected error: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Unexpected error in run: {str(e)}\n")
            return False

    def update_report_file(self):
        try:
            with open(self.report_file, 'a') as f:
                f.write(f"\n=== UPDATE: {time.ctime()} ===\n")
                exact_matches = sum(1 for addr in self.target_addresses
                                   if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
                f.write(f"Exact matches found: {exact_matches}/{len(self.target_addresses)}\n")
                if self.best_matches:
                    sorted_matches = sorted(
                        [(addr, match) for addr, match in self.best_matches.items()],
                        key=lambda x: x[1]['hamming']
                    )
                    f.write(f"\nTOP 5 CLOSEST MATCHES:\n")
                    for i, (addr, match) in enumerate(sorted_matches[:5]):
                        if match['exact_match']:
                            f.write(f"#{i+1}: {addr} EXACT MATCH!\n")
                            f.write(f"    Private Key: {match['private_key']}\n")
                            f.write(f"    Entropy Level: {match['entropy']:.2f}/10\n")
                        else:
                            closeness = (160 - match['hamming']) / 160 * 100
                            f.write(f"#{i+1}: {addr} (Hamming: {match['hamming']}, {closeness:.2f}% similar)\n")
                            f.write(f"    Private Key: {match['private_key']}\n")
                            f.write(f"    Entropy Level: {match['entropy']:.2f}/10\n")
                            if match['hamming'] < 38:
                                f.write(f"    **BRUTE-FORCE CANDIDATE**\n")
                            elif match['hamming'] <= 45:
                                f.write(f"    **CLOSE MATCH**\n")
        except Exception as e:
            print(f"{Fore.RED}Error updating report file: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Error updating report file: {str(e)}\n")

    def display_progress(self):
        if self.silent_mode:
            return
        print(f"\n{Fore.CYAN}Progress Update:")
        exact_matches = sum(1 for addr in self.target_addresses
                           if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
        print(f"{Fore.GREEN}Exact matches found: {exact_matches}/{len(self.target_addresses)}")
        if self.best_matches:
            sorted_matches = sorted(
                [(addr, match) for addr, match in self.best_matches.items()],
                key=lambda x: x[1]['hamming']
            )
            print(f"\n{Fore.CYAN}Top 3 Closest Matches:")
            for i, (addr, match) in enumerate(sorted_matches[:3]):
                if match['exact_match']:
                    print(f"{Fore.GREEN}#{i+1}: {addr[:8]}...{addr[-8:]} EXACT MATCH!")
                    print(f"{Fore.GREEN}    Private Key: {match['private_key']}")
                    print(f"{Fore.GREEN}    Entropy Level: {match['entropy']:.2f}/10")
                else:
                    closeness = (160 - match['hamming']) / 160 * 100
                    print(f"{Fore.YELLOW}#{i+1}: {addr[:8]}...{addr[-8:]} (Hamming: {match['hamming']}, {closeness:.2f}% similar)")
                    print(f"{Fore.YELLOW}    Private Key: {match['private_key']}")
                    print(f"{Fore.YELLOW}    Entropy Level: {match['entropy']:.2f}/10")
                    if match['hamming'] < 38:
                        print(f"{Fore.YELLOW}    **BRUTE-FORCE CANDIDATE**")
                    elif match['hamming'] <= 45:
                        print(f"{Fore.YELLOW}    **CLOSE MATCH**")

    def generate_final_report(self):
        try:
            with open(self.report_file, 'a') as f:
                f.write(f"\n\n=========== FINAL COUNTER ANALYSIS REPORT ===========\n")
                f.write(f"Generated at: {time.ctime()}\n\n")
                for addr in self.target_addresses:
                    if addr not in self.best_matches:
                        continue
                    f.write(f"\nAddress: {addr}\n")
                    match = self.best_matches[addr]
                    if match['exact_match']:
                        f.write(f"EXACT MATCH FOUND!\n")
                        f.write(f"Private Key: {match['private_key']}\n")
                        f.write(f"Entropy Level: {match['entropy']:.2f}/10\n")
                    else:
                        closeness = (160 - match['hamming']) / 160 * 100
                        f.write(f"Best Match: (Hamming: {match['hamming']}, {closeness:.2f}% similar)\n")
                        f.write(f"Private Key: {match['private_key']}\n")
                        f.write(f"Entropy Level: {match['entropy']:.2f}/10\n")
                        if match['hamming'] < 38:
                            f.write(f"**BRUTE-FORCE CANDIDATE**\n")
                        elif match['hamming'] <= 45:
                            f.write(f"**CLOSE MATCH**\n")
            with open("best_matches.csv", "w") as csv_file:
                csv_file.write("Address,Private Key,Hamming Distance,Similarity %,Entropy Level,Exact Match\n")
                for addr, match in self.best_matches.items():
                    similarity = (160 - match['hamming']) / 160 * 100 if not match['exact_match'] else 100
                    csv_file.write(f"{addr},{match['private_key']},{match['hamming']},{similarity:.2f},{match['entropy']:.2f},{match['exact_match']}\n")
            print(f"{Fore.GREEN}Best matches also saved to best_matches.csv")
        except Exception as e:
            print(f"{Fore.RED}Error generating final report: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Error generating final report: {str(e)}\n")
            print(f"\n{Fore.CYAN}===== COUNTER ANALYSIS SUMMARY =====")
            exact_matches = sum(1 for addr in self.target_addresses
                               if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
            print(f"{Fore.GREEN}Exact matches found: {exact_matches}/{len(self.target_addresses)}")
            close_matches = [(addr, match) for addr, match in self.best_matches.items()
                            if not match.get('exact_match', False) and match['hamming'] < 38]
            if close_matches:
                print(f"{Fore.YELLOW}Found {len(close_matches)} brute-force candidates (Hamming < 38)")
                print(f"{Fore.YELLOW}Check the report file for details")

def main():
    print(f"""
{Fore.CYAN}╔══════════════════════════════════════════════════════════════════╗
{Fore.CYAN}║                                                                  ║
{Fore.CYAN}║  {Fore.GREEN}BITCOIN KEY COUNTER ANALYZER{Fore.CYAN}                                ║
{Fore.CYAN}║  {Fore.YELLOW}Test Sequential Keys Starting from k=1{Fore.CYAN}                     ║
{Fore.CYAN}╚══════════════════════════════════════════════════════════════════╝
    """)
    analyzer = FastCounterAnalyzer()
    iterations = 5000
    batch_factor = 2.0
    print(f"{Fore.CYAN}Running with {iterations} iterations and batch size factor {batch_factor}")
    success = analyzer.run(iterations=iterations, batch_factor=batch_factor)
    if not success:
        print(f"{Fore.RED}Analysis failed. Check error_log.txt for details.")

if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║  BITCOIN KEY COUNTER ANALYZER                                ║
║  Test Sequential Keys Starting from k=1                     ║
╚══════════════════════════════════════════════════════════════════╝
    
Running with 5000 iterations and batch size factor 2.0
Processing addresses...
Loaded 999 valid addresses.
Starting counter-based key analysis for 999 addresses
Using 8 CPU cores for maximum speed
Press Ctrl+C to pause analysis and save progress
Report will be saved to counter_report.txt

Iteration 1/5000 - Testing keys 1 to 40000

Progress Update:
Exact matches found: 0/999

Top 3 Closest Matches:
#1: 1PAv6QCV...z7BqkvZU (Hamming: 48, 70.00% similar)
    Private Key: 0000000000000000000000000000000000000000000000000000000000006b73
    Entropy Level: 0.00/10
Progress saved.

Iteration 51/5000 - Testing keys 2000001 to 2040000

Progress Update:
Exact 

SystemExit: 0

In [ ]:
#!/usr/bin/env python3
# Bitcoin Private Key Breaker - Brute-Force Around Close Keys from close_keys.txt

# Install required packages
!pip install -q base58 colorama tqdm ecdsa pycryptodome

import hashlib
import base58
import os
import time
import random
from colorama import Fore, init
from tqdm.notebook import tqdm
from google.colab import files
import signal
import sys
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import pickle
import binascii
import ecdsa
from Crypto.Hash import RIPEMD160

# Initialize colorama
init(autoreset=True)

# SECP256k1 curve order (n)
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Global variables for child processes
TARGET_ADDRESSES = []
TARGET_HASH160S = {}

def generate_address(private_key_int):
    try:
        # Validate key range
        if not (0 < private_key_int < CURVE_ORDER):
            with open('error_log.txt', 'a') as f:
                f.write(f"Invalid key in generate_address: {private_key_int} (out of range 1 to {CURVE_ORDER-1})\n")
            return None, None
        private_key_hex = format(private_key_int, '064x')
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()
        public_key = b'\x04' + vk.to_string()
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160_hasher = RIPEMD160.new()
        ripemd160_hasher.update(sha256_hash)
        hash160 = ripemd160_hasher.digest()
        versioned_hash = b'\x00' + hash160
        checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
        address_bytes = versioned_hash + checksum
        address = base58.b58encode(address_bytes).decode('utf-8')
        return address, hash160
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error in generate_address for key {private_key_int}: {str(e)}\n")
        return None, None

def hamming_distance(hash1, hash2):
    try:
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except:
        return 160

def read_close_keys():
    close_keys = []
    try:
        with open('close_keys.txt', 'r') as f:
            lines = f.readlines()
            header = lines[0].strip().split(',')
            if header != ['Address', 'Private Key', 'Hamming Distance', 'Entropy Level']:
                raise ValueError("Invalid close_keys.txt header")
            for line in lines[1:]:
                addr, priv_key, hamming, entropy = line.strip().split(',')
                close_keys.append({
                    'address': addr,
                    'private_key': priv_key,
                    'hamming': int(hamming),
                    'entropy': float(entropy)
                })
        # Sort by Hamming distance, prioritizing lowest
        close_keys.sort(key=lambda x: x['hamming'])
        return close_keys
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error reading close_keys.txt: {str(e)}\n")
        return []

def process_key_batch(batch_data):
    import random
    base_key, offsets, target_addr, target_hash160 = batch_data
    results = []
    try:
        for offset in offsets:
            # Generate candidate key
            key_int = (base_key + offset) % (CURVE_ORDER - 1)
            if key_int == 0:
                key_int = 1
            address, hash160 = generate_address(key_int)
            if not address or not hash160:
                continue
            if random.random() < 0.1:  # Uniform 10% sampling
                hamming = hamming_distance(hash160, target_hash160)
                if hamming < 50:
                    result = {
                        'target': target_addr,
                        'key_int': key_int,
                        'private_key': format(key_int, '064x'),
                        'hamming': hamming,
                        'exact_match': False,
                        'entropy': 0.0
                    }
                    results.append(result)
                    if hamming <= 38:
                        with open('breaker_results.txt', 'a') as f:
                            f.write(f"{target_addr},{result['private_key']},{hamming},0.00\n")
            if address == target_addr:
                result = {
                    'target': target_addr,
                    'key_int': key_int,
                    'private_key': format(key_int, '064x'),
                    'hamming': 0,
                    'exact_match': True,
                    'entropy': 0.0
                }
                results.append(result)
                with open('breaker_results.txt', 'a') as f:
                    f.write(f"EXACT MATCH,{target_addr},{result['private_key']},0,0.00\n")
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error in process_key_batch for base_key {base_key}: {str(e)}\n")
    return results

class FastKeyBreaker:
    def __init__(self):
        self.target_addresses = []
        self.results = {}
        self.best_matches = {}
        self.checkpoint_file = 'key_breaker.pkl'
        self.cores = multiprocessing.cpu_count()
        self.running = True
        self.report_file = 'breaker_report.txt'
        self.silent_mode = False
        signal.signal(signal.SIGINT, self.handle_interrupt)

    def handle_interrupt(self, sig, frame):
        print(f"\n{Fore.YELLOW}Interrupt received, saving progress...")
        self.running = False
        self.save_checkpoint()
        time.sleep(1)
        sys.exit(0)

    def load_addresses(self):
        print(f"{Fore.CYAN}Processing addresses...")
        try:
            with open("addresses.txt", 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]
        except:
            try:
                with open("addresses (1).txt", 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]
            except:
                print(f"{Fore.CYAN}Upload a text file with Bitcoin addresses (one per line)")
                uploaded = files.upload()
                if not uploaded:
                    print(f"{Fore.RED}No file uploaded. Exiting.")
                    return False
                filename = list(uploaded.keys())[0]
                with open(filename, 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]

        valid_addresses = []
        global TARGET_ADDRESSES, TARGET_HASH160S
        TARGET_ADDRESSES = []
        TARGET_HASH160S = {}
        for addr in addresses:
            if not addr.startswith('1'):
                continue
            try:
                decoded = base58.b58decode(addr)
                if len(decoded) != 25:
                    continue
                hash160 = decoded[1:-4]
                valid_addresses.append(addr)
                TARGET_ADDRESSES.append(addr)
                TARGET_HASH160S[addr] = hash160
                self.results[addr] = []
            except:
                continue
        if not valid_addresses:
            print(f"{Fore.RED}No valid P2PKH addresses found.")
            return False
        self.target_addresses = valid_addresses
        print(f"{Fore.GREEN}Loaded {len(valid_addresses)} valid addresses.")
        return True

    def save_checkpoint(self):
        checkpoint_data = {
            'targets': self.target_addresses,
            'results': self.results,
            'best_matches': self.best_matches
        }
        try:
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
            if not self.silent_mode:
                print(f"{Fore.GREEN}Progress saved.")
            return True
        except Exception as e:
            print(f"{Fore.RED}Error saving checkpoint: {str(e)}")
            return False

    def load_checkpoint(self):
        if not os.path.exists(self.checkpoint_file):
            return False
        try:
            with open(self.checkpoint_file, 'rb') as f:
                data = pickle.load(f)
            self.target_addresses = data['targets']
            self.results = data['results']
            self.best_matches = data.get('best_matches', {})
            global TARGET_ADDRESSES, TARGET_HASH160S
            TARGET_ADDRESSES = self.target_addresses.copy()
            TARGET_HASH160S = {}
            for addr in self.target_addresses:
                try:
                    decoded = base58.b58decode(addr)
                    hash160 = decoded[1:-4]
                    TARGET_HASH160S[addr] = hash160
                except:
                    pass
            print(f"{Fore.GREEN}Loaded checkpoint with {len(self.target_addresses)} addresses.")
            return True
        except Exception as e:
            print(f"{Fore.RED}Error loading checkpoint: {str(e)}")
            return False

    def run(self, iterations=5000, batch_factor=2.0):
        if not self.load_checkpoint():
            if not self.load_addresses():
                return False

        close_keys = read_close_keys()
        if not close_keys:
            print(f"{Fore.RED}No close keys found in close_keys.txt. Exiting.")
            return False

        print(f"{Fore.GREEN}Starting key breaker analysis for {len(self.target_addresses)} addresses")
        print(f"{Fore.GREEN}Using {len(close_keys)} close keys from close_keys.txt")
        print(f"{Fore.GREEN}Using {self.cores} CPU cores for maximum speed")
        print(f"{Fore.GREEN}Press Ctrl+C to pause analysis and save progress")
        print(f"{Fore.GREEN}Report will be saved to {self.report_file}")

        start_time = time.time()

        try:
            with open(self.report_file, 'w') as f:
                f.write(f"=== BITCOIN KEY BREAKER ANALYSIS REPORT ===\n")
                f.write(f"Started at: {time.ctime()}\n")
                f.write(f"Analyzing {len(self.target_addresses)} addresses with {len(close_keys)} base keys\n\n")

            with open('breaker_results.txt', 'w') as f:
                f.write("Address,Private Key,Hamming Distance,Entropy Level\n")
            with open('error_log.txt', 'w') as f:
                f.write("Error Log\n")

            for key_idx, key_data in enumerate(close_keys):
                if not self.running:
                    break

                addr = key_data['address']
                base_key = int(key_data['private_key'], 16)
                hamming = key_data['hamming']
                target_hash160 = TARGET_HASH160S.get(addr)
                if not target_hash160:
                    with open('error_log.txt', 'a') as f:
                        f.write(f"Target address {addr} not found in TARGET_HASH160S\n")
                    continue

                print(f"\n{Fore.CYAN}Processing key {key_idx+1}/{len(close_keys)} for {addr[:8]}... (Hamming: {hamming})")

                batch_size = int(2500 * batch_factor)  # Keys per batch
                for iteration in range(iterations):
                    exact_matches = sum(1 for addr in self.target_addresses
                                       if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
                    if exact_matches == len(self.target_addresses):
                        print(f"{Fore.GREEN}All private keys recovered successfully!")
                        break

                    # Generate offsets: positive and negative, plus bit flips
                    offsets = []
                    max_offset = 1000000  # ~2^20
                    for i in range(-batch_size//2, batch_size//2):
                        offsets.append(i)
                    # Add bit flips (1–5 bits)
                    for _ in range(batch_size//10):
                        bit_pos = random.randint(0, 255)
                        offsets.append(1 << bit_pos)
                        if len(offsets) < batch_size:
                            bit_pos2 = random.randint(0, 255)
                            offsets.append((1 << bit_pos) ^ (1 << bit_pos2))

                    batches = [(base_key, offsets[i:i+batch_size//self.cores], addr, target_hash160)
                               for i in range(0, batch_size, batch_size//self.cores)]

                    if iteration % 50 == 0 or iteration == 0:
                        print(f"\n{Fore.CYAN}Iteration {iteration+1}/{iterations} for {addr[:8]}...")
                    else:
                        self.silent_mode = True

                    with ProcessPoolExecutor(max_workers=self.cores) as executor:
                        futures = [executor.submit(process_key_batch, batch) for batch in batches]
                        for future in futures:
                            try:
                                batch_results = future.result()
                                for result in batch_results:
                                    if not isinstance(result, dict):
                                        with open('error_log.txt', 'a') as f:
                                            f.write(f"Invalid result type in batch_results: {type(result)}\n")
                                        continue
                                    result_addr = result['target']
                                    self.results[result_addr].append(result)
                                    if result_addr not in self.best_matches or result['hamming'] < self.best_matches[result_addr]['hamming']:
                                        self.best_matches[result_addr] = result
                                        if not self.silent_mode or result['hamming'] < 38:
                                            if result['exact_match']:
                                                print(f"{Fore.GREEN}EXACT MATCH FOUND for {result_addr[:8]}...{result_addr[-4:]}")
                                            elif result['hamming'] < 38:
                                                print(f"{Fore.YELLOW}BRUTE-FORCE RANGE for {result_addr[:8]}...{result_addr[-4:]} (Hamming: {result['hamming']})")
                            except Exception as e:
                                with open('error_log.txt', 'a') as f:
                                    f.write(f"Error processing future result: {str(e)}\n")

                    if iteration % 50 == 0 or iteration == iterations-1:
                        self.silent_mode = False
                        self.display_progress()
                        self.save_checkpoint()
                        self.update_report_file()
                    else:
                        self.save_checkpoint()

            self.silent_mode = False
            self.save_checkpoint()
            self.generate_final_report()

            elapsed = time.time() - start_time
            print(f"{Fore.GREEN}Analysis completed in {elapsed:.2f} seconds")
            print(f"{Fore.GREEN}Final report saved to {self.report_file}")
            print(f"{Fore.GREEN}Results saved to breaker_results.txt")

            try:
                files.download(self.report_file)
                print(f"{Fore.GREEN}Report downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download report automatically. Please download it manually.")

            try:
                files.download('best_matches.csv')
                print(f"{Fore.GREEN}Best matches CSV downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download CSV automatically. Please download it manually.")

            try:
                files.download('breaker_results.txt')
                print(f"{Fore.GREEN}Breaker results downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download breaker_results.txt automatically. Please download it manually.")

            try:
                files.download('error_log.txt')
                print(f"{Fore.GREEN}Error log downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download error_log.txt automatically. Please download it manually.")

            return True

        except KeyboardInterrupt:
            print(f"\n{Fore.YELLOW}Analysis interrupted. Saving progress...")
            self.save_checkpoint()
            self.update_report_file()
            return False
        except Exception as e:
            print(f"{Fore.RED}Unexpected error: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Unexpected error in run: {str(e)}\n")
            return False

    def update_report_file(self):
        try:
            with open(self.report_file, 'a') as f:
                f.write(f"\n=== UPDATE: {time.ctime()} ===\n")
                exact_matches = sum(1 for addr in self.target_addresses
                                   if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
                f.write(f"Exact matches found: {exact_matches}/{len(self.target_addresses)}\n")
                if self.best_matches:
                    sorted_matches = sorted(
                        [(addr, match) for addr, match in self.best_matches.items()],
                        key=lambda x: x[1]['hamming']
                    )
                    f.write(f"\nTOP 5 CLOSEST MATCHES:\n")
                    for i, (addr, match) in enumerate(sorted_matches[:5]):
                        if match['exact_match']:
                            f.write(f"#{i+1}: {addr} EXACT MATCH!\n")
                            f.write(f"    Private Key: {match['private_key']}\n")
                            f.write(f"    Entropy Level: {match['entropy']:.2f}/10\n")
                        else:
                            closeness = (160 - match['hamming']) / 160 * 100
                            f.write(f"#{i+1}: {addr} (Hamming: {match['hamming']}, {closeness:.2f}% similar)\n")
                            f.write(f"    Private Key: {match['private_key']}\n")
                            f.write(f"    Entropy Level: {match['entropy']:.2f}/10\n")
                            if match['hamming'] < 38:
                                f.write(f"    **BRUTE-FORCE CANDIDATE**\n")
                            elif match['hamming'] <= 45:
                                f.write(f"    **CLOSE MATCH**\n")
        except Exception as e:
            print(f"{Fore.RED}Error updating report file: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Error updating report file: {str(e)}\n")

    def display_progress(self):
        if self.silent_mode:
            return
        print(f"\n{Fore.CYAN}Progress Update:")
        exact_matches = sum(1 for addr in self.target_addresses
                           if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
        print(f"{Fore.GREEN}Exact matches found: {exact_matches}/{len(self.target_addresses)}")
        if self.best_matches:
            sorted_matches = sorted(
                [(addr, match) for addr, match in self.best_matches.items()],
                key=lambda x: x[1]['hamming']
            )
            print(f"\n{Fore.CYAN}Top 3 Closest Matches:")
            for i, (addr, match) in enumerate(sorted_matches[:3]):
                if match['exact_match']:
                    print(f"{Fore.GREEN}#{i+1}: {addr[:8]}...{addr[-8:]} EXACT MATCH!")
                    print(f"{Fore.GREEN}    Private Key: {match['private_key']}")
                    print(f"{Fore.GREEN}    Entropy Level: {match['entropy']:.2f}/10")
                else:
                    closeness = (160 - match['hamming']) / 160 * 100
                    print(f"{Fore.YELLOW}#{i+1}: {addr[:8]}...{addr[-8:]} (Hamming: {match['hamming']}, {closeness:.2f}% similar)")
                    print(f"{Fore.YELLOW}    Private Key: {match['private_key']}")
                    print(f"{Fore.YELLOW}    Entropy Level: {match['entropy']:.2f}/10")
                    if match['hamming'] < 38:
                        print(f"{Fore.YELLOW}    **BRUTE-FORCE CANDIDATE**")
                    elif match['hamming'] <= 45:
                        print(f"{Fore.YELLOW}    **CLOSE MATCH**")

    def generate_final_report(self):
        try:
            with open(self.report_file, 'a') as f:
                f.write(f"\n\n=========== FINAL KEY BREAKER ANALYSIS REPORT ===========\n")
                f.write(f"Generated at: {time.ctime()}\n\n")
                for addr in self.target_addresses:
                    if addr not in self.best_matches:
                        continue
                    f.write(f"\nAddress: {addr}\n")
                    match = self.best_matches[addr]
                    if match['exact_match']:
                        f.write(f"EXACT MATCH FOUND!\n")
                        f.write(f"Private Key: {match['private_key']}\n")
                        f.write(f"Entropy Level: {match['entropy']:.2f}/10\n")
                    else:
                        closeness = (160 - match['hamming']) / 160 * 100
                        f.write(f"Best Match: (Hamming: {match['hamming']}, {closeness:.2f}% similar)\n")
                        f.write(f"Private Key: {match['private_key']}\n")
                        f.write(f"Entropy Level: {match['entropy']:.2f}/10\n")
                        if match['hamming'] < 38:
                            f.write(f"**BRUTE-FORCE CANDIDATE**\n")
                        elif match['hamming'] <= 45:
                            f.write(f"**CLOSE MATCH**\n")
            with open("best_matches.csv", "w") as csv_file:
                csv_file.write("Address,Private Key,Hamming Distance,Similarity %,Entropy Level,Exact Match\n")
                for addr, match in self.best_matches.items():
                    similarity = (160 - match['hamming']) / 160 * 100 if not match['exact_match'] else 100
                    csv_file.write(f"{addr},{match['private_key']},{match['hamming']},{similarity:.2f},{match['entropy']:.2f},{match['exact_match']}\n")
            print(f"{Fore.GREEN}Best matches also saved to best_matches.csv")
        except Exception as e:
            print(f"{Fore.RED}Error generating final report: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Error generating final report: {str(e)}\n")
            print(f"\n{Fore.CYAN}===== KEY BREAKER ANALYSIS SUMMARY =====")
            exact_matches = sum(1 for addr in self.target_addresses
                               if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
            print(f"{Fore.GREEN}Exact matches found: {exact_matches}/{len(self.target_addresses)}")
            close_matches = [(addr, match) for addr, match in self.best_matches.items()
                            if not match.get('exact_match', False) and match['hamming'] < 38]
            if close_matches:
                print(f"{Fore.YELLOW}Found {len(close_matches)} brute-force candidates (Hamming < 38)")
                print(f"{Fore.YELLOW}Check the report file for details")

def main():
    print(f"""
{Fore.CYAN}╔══════════════════════════════════════════════════════════════════╗
{Fore.CYAN}║                                                                  ║
{Fore.CYAN}║  {Fore.GREEN}BITCOIN PRIVATE KEY BREAKER{Fore.CYAN}                                 ║
{Fore.CYAN}║  {Fore.YELLOW}Brute-Force Around Close Keys from close_keys.txt{Fore.CYAN}         ║
{Fore.CYAN}╚══════════════════════════════════════════════════════════════════╝
    """)
    breaker = FastKeyBreaker()
    iterations = 5000
    batch_factor = 2.0
    print(f"{Fore.CYAN}Running with {iterations} iterations and batch size factor {batch_factor}")
    success = breaker.run(iterations=iterations, batch_factor=batch_factor)
    if not success:
        print(f"{Fore.RED}Analysis failed. Check error_log.txt for details.")

if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║  BITCOIN PRIVATE KEY BREAKER                                 ║
║  Brute-Force Around Close Keys from close_keys.txt         ║
╚══════════════════════════════════════════════════════════════════╝
    
Running with 5000 iterations and batch size factor 2.0
Processing addresses...
Loaded 999 valid addresses.
Starting key breaker analysis for 999 addresses
Using 105 close keys from close_keys.txt
Using 8 CPU cores for maximum speed
Press Ctrl+C to pause analysis and save progress
Report will be saved to breaker_report.txt

Processing key 1/105 for 1Mp5TX53... (Hamming: 39)

Iteration 1/5000 for 1Mp5TX53...

Progress Update:
Exact matches found: 0/999
Progress saved.

Iteration 51/5000 for 1Mp5TX53...

Progress Update:
Exact matches found: 0/999

Top 3 Closest Matches:
#1: 1Mp5TX53...7i2gah13 (Hamming: 39, 75.62% similar)
    Private Key: 0000000000000

SystemExit: 0

In [ ]:
#!/usr/bin/env python3
# Bitcoin Private Key Breaker - Liberal Exploration Around Close Keys (Fixed Undefined Variables)

# Install required packages
!pip install -q base58 colorama tqdm ecdsa pycryptodome

import hashlib
import base58
import os
import time
import random
from colorama import Fore, init
from tqdm.notebook import tqdm
from google.colab import files
import signal
import sys
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
import pickle
import binascii
import ecdsa
from Crypto.Hash import RIPEMD160

# Initialize colorama
init(autoreset=True)

# SECP256k1 curve order (n)
CURVE_ORDER = 115792089237316195423570985008687907852837564279074904382605163141518161494337

# Global variables for child processes
TARGET_ADDRESSES = []
TARGET_HASH160S = {}

def generate_address(private_key_int):
    try:
        # Validate key range
        if not (0 < private_key_int < CURVE_ORDER):
            with open('error_log.txt', 'a') as f:
                f.write(f"Invalid key in generate_address: {private_key_int} (out of range 1 to {CURVE_ORDER-1})\n")
            return None, None
        private_key_hex = format(private_key_int, '064x')
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()
        public_key = b'\x04' + vk.to_string()
        sha256_hash = hashlib.sha256(public_key).digest()
        ripemd160_hasher = RIPEMD160.new()
        ripemd160_hasher.update(sha256_hash)
        hash160 = ripemd160_hasher.digest()
        versioned_hash = b'\x00' + hash160
        checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
        address_bytes = versioned_hash + checksum
        address = base58.b58encode(address_bytes).decode('utf-8')
        return address, hash160
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error in generate_address for key {private_key_int}: {str(e)}\n")
        return None, None

def hamming_distance(hash1, hash2):
    try:
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except:
        return 160

def read_close_keys(max_keys=10):
    close_keys = []
    try:
        with open('close_keys.txt', 'r') as f:
            lines = f.readlines()
            header = lines[0].strip().split(',')
            if header != ['Address', 'Private Key', 'Hamming Distance', 'Entropy Level']:
                raise ValueError("Invalid close_keys.txt header")
            for line in lines[1:]:
                addr, priv_key, hamming, entropy = line.strip().split(',')
                close_keys.append({
                    'address': addr,
                    'private_key': priv_key,
                    'hamming': int(hamming),
                    'entropy': float(entropy)
                })
        # Sort by Hamming distance, take top max_keys (prioritize Hamming <=41)
        close_keys.sort(key=lambda x: x['hamming'])
        return close_keys[:max_keys]
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error reading close_keys.txt: {str(e)}\n")
        return []

def extract_significant_digits(priv_key_hex):
    # Remove leading zeros, return non-zero hex digits
    return priv_key_hex.lstrip('0') or '0'

def generate_candidate_key(base_key, significant_digits, variation_type, shift=0, flip_bits=0):
    try:
        if variation_type == 'offset':
            # Apply offset to base key
            key_int = (base_key + shift) % (CURVE_ORDER - 1)
        elif variation_type == 'shift':
            # Shift significant digits left by shift bits
            sig_int = int(significant_digits, 16)
            key_int = (sig_int << shift) % (CURVE_ORDER - 1)
        elif variation_type == 'fewer_zeros':
            # Insert random digits, keep significant digits
            prefix = ''.join(random.choice('0123456789abcdef') for _ in range(random.randint(0, 10)))
            suffix = ''.join(random.choice('0123456789abcdef') for _ in range(random.randint(0, 10)))
            key_str = (prefix + significant_digits + suffix)[:64].ljust(64, '0')
            key_int = int(key_str, 16) % (CURVE_ORDER - 1)
        elif variation_type == 'bit_flip':
            # Flip 1–10 bits
            key_int = base_key
            for _ in range(flip_bits):
                bit_pos = random.randint(0, 255)
                key_int ^= (1 << bit_pos)
            key_int = key_int % (CURVE_ORDER - 1)
        else:  # random_pattern
            # Add random low-entropy pattern around significant digits
            pattern = random.choice(['1111', '5555', 'aaaa', 'ffff']) * random.randint(1, 4)
            key_str = (significant_digits + pattern)[:64].ljust(64, '0')
            key_int = int(key_str, 16) % (CURVE_ORDER - 1)

        # Ensure key is within valid range
        if key_int == 0:
            key_int = 1
        return key_int
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error generating candidate key: {str(e)}\n")
        return random.randint(1, CURVE_ORDER - 1)

def process_key_batch(batch_data):
    import random
    base_key, significant_digits, target_addr, target_hash160, batch_size = batch_data
    results = []
    try:
        with open('breaker_results.txt', 'a') as f:
            for _ in range(batch_size):
                variation = random.choice(['offset', 'shift', 'fewer_zeros', 'bit_flip', 'random_pattern'])
                # Initialize shift and flip_bits with defaults
                shift = 0
                flip_bits = 0
                if variation == 'offset':
                    shift = random.randint(-2**30, 2**30)  # Large offset range
                elif variation == 'shift':
                    shift = random.randint(0, 240)  # Shift up to 240 bits
                elif variation == 'bit_flip':
                    flip_bits = random.randint(1, 10)  # Flip 1–10 bits
                key_int = generate_candidate_key(base_key, significant_digits, variation, shift, flip_bits)
                address, hash160 = generate_address(key_int)
                if not address or not hash160:
                    continue
                if random.random() < 0.1:  # Uniform 10% sampling
                    hamming = hamming_distance(hash160, target_hash160)
                    if hamming < 50:
                        result = {
                            'target': target_addr,
                            'key_int': key_int,
                            'private_key': format(key_int, '064x'),
                            'hamming': hamming,
                            'exact_match': False,
                            'entropy': 0.0
                        }
                        results.append(result)
                        if hamming <= 38:
                            f.write(f"{target_addr},{result['private_key']},{hamming},0.00\n")
                if address == target_addr:
                    result = {
                        'target': target_addr,
                        'key_int': key_int,
                        'private_key': format(key_int, '064x'),
                        'hamming': 0,
                        'exact_match': True,
                        'entropy': 0.0
                    }
                    results.append(result)
                    f.write(f"EXACT MATCH,{target_addr},{result['private_key']},0,0.00\n")
    except Exception as e:
        with open('error_log.txt', 'a') as f:
            f.write(f"Error in process_key_batch for base_key {base_key}: {str(e)}\n")
    return results

class FastKeyBreaker:
    def __init__(self):
        self.target_addresses = []
        self.results = {}
        self.best_matches = {}
        self.checkpoint_file = 'key_breaker.pkl'
        self.cores = multiprocessing.cpu_count()
        self.running = True
        self.report_file = 'breaker_report.txt'
        self.silent_mode = False
        signal.signal(signal.SIGINT, self.handle_interrupt)

    def handle_interrupt(self, sig, frame):
        print(f"\n{Fore.YELLOW}Interrupt received, saving progress...")
        self.running = False
        self.save_checkpoint()
        time.sleep(1)
        sys.exit(0)

    def load_addresses(self):
        print(f"{Fore.CYAN}Processing addresses...")
        try:
            with open("addresses.txt", 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]
        except:
            try:
                with open("addresses (1).txt", 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]
            except:
                print(f"{Fore.CYAN}Upload a text file with Bitcoin addresses (one per line)")
                uploaded = files.upload()
                if not uploaded:
                    print(f"{Fore.RED}No file uploaded. Exiting.")
                    return False
                filename = list(uploaded.keys())[0]
                with open(filename, 'r') as f:
                    addresses = [line.strip() for line in f if line.strip()]

        valid_addresses = []
        global TARGET_ADDRESSES, TARGET_HASH160S
        TARGET_ADDRESSES = []
        TARGET_HASH160S = {}
        for addr in addresses:
            if not addr.startswith('1'):
                continue
            try:
                decoded = base58.b58decode(addr)
                if len(decoded) != 25:
                    continue
                hash160 = decoded[1:-4]
                valid_addresses.append(addr)
                TARGET_ADDRESSES.append(addr)
                TARGET_HASH160S[addr] = hash160
                self.results[addr] = []
            except:
                continue
        if not valid_addresses:
            print(f"{Fore.RED}No valid P2PKH addresses found.")
            return False
        self.target_addresses = valid_addresses
        print(f"{Fore.GREEN}Loaded {len(valid_addresses)} valid addresses.")
        return True

    def save_checkpoint(self):
        checkpoint_data = {
            'targets': self.target_addresses,
            'results': self.results,
            'best_matches': self.best_matches
        }
        try:
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint_data, f)
            if not self.silent_mode:
                print(f"{Fore.GREEN}Progress saved.")
            return True
        except Exception as e:
            print(f"{Fore.RED}Error saving checkpoint: {str(e)}")
            return False

    def load_checkpoint(self):
        if not os.path.exists(self.checkpoint_file):
            return False
        try:
            with open(self.checkpoint_file, 'rb') as f:
                data = pickle.load(f)
            self.target_addresses = data['targets']
            self.results = data['results']
            self.best_matches = data.get('best_matches', {})
            global TARGET_ADDRESSES, TARGET_HASH160S
            TARGET_ADDRESSES = self.target_addresses.copy()
            TARGET_HASH160S = {}
            for addr in self.target_addresses:
                try:
                    decoded = base58.b58decode(addr)
                    hash160 = decoded[1:-4]
                    TARGET_HASH160S[addr] = hash160
                except:
                    pass
            print(f"{Fore.GREEN}Loaded checkpoint with {len(self.target_addresses)} addresses.")
            return True
        except Exception as e:
            print(f"{Fore.RED}Error loading checkpoint: {str(e)}")
            return False

    def run(self, iterations=5000, batch_factor=2.0):
        if not self.load_checkpoint():
            if not self.load_addresses():
                return False

        close_keys = read_close_keys(max_keys=10)
        if not close_keys:
            print(f"{Fore.RED}No close keys found in close_keys.txt. Exiting.")
            return False

        print(f"{Fore.GREEN}Starting key breaker analysis for {len(self.target_addresses)} addresses")
        print(f"{Fore.GREEN}Using {len(close_keys)} close keys from close_keys.txt (Hamming {close_keys[0]['hamming']}–{close_keys[-1]['hamming']})")
        print(f"{Fore.GREEN}Using {self.cores} CPU cores for maximum speed")
        print(f"{Fore.GREEN}Press Ctrl+C to pause analysis and save progress")
        print(f"{Fore.GREEN}Report will be saved to {self.report_file}")

        start_time = time.time()

        try:
            with open(self.report_file, 'w') as f:
                f.write(f"=== BITCOIN KEY BREAKER ANALYSIS REPORT ===\n")
                f.write(f"Started at: {time.ctime()}\n")
                f.write(f"Analyzing {len(self.target_addresses)} addresses with {len(close_keys)} base keys\n\n")

            with open('breaker_results.txt', 'w') as f:
                f.write("Address,Private Key,Hamming Distance,Entropy Level\n")
            with open('error_log.txt', 'w') as f:
                f.write("Error Log\n")

            for key_idx, key_data in enumerate(close_keys):
                if not self.running:
                    break

                addr = key_data['address']
                base_key = int(key_data['private_key'], 16)
                hamming = key_data['hamming']
                significant_digits = extract_significant_digits(key_data['private_key'])
                target_hash160 = TARGET_HASH160S.get(addr)
                if not target_hash160:
                    with open('error_log.txt', 'a') as f:
                        f.write(f"Target address {addr} not found in TARGET_HASH160S\n")
                    continue

                print(f"\n{Fore.CYAN}Processing key {key_idx+1}/{len(close_keys)} for {addr[:8]}... (Hamming: {hamming}, Significant: {significant_digits})")

                batch_size = int(2500 * batch_factor)  # Keys per batch
                for iteration in range(iterations):
                    exact_matches = sum(1 for addr in self.target_addresses
                                       if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
                    if exact_matches == len(self.target_addresses):
                        print(f"{Fore.GREEN}All private keys recovered successfully!")
                        break

                    batches = [(base_key, significant_digits, addr, target_hash160, batch_size//self.cores)
                               for _ in range(self.cores)]

                    if iteration % 50 == 0 or iteration == 0:
                        print(f"\n{Fore.CYAN}Iteration {iteration+1}/{iterations} for {addr[:8]}...")
                    else:
                        self.silent_mode = True

                    with ProcessPoolExecutor(max_workers=self.cores) as executor:
                        futures = [executor.submit(process_key_batch, batch) for batch in batches]
                        for future in futures:
                            try:
                                batch_results = future.result()
                                for result in batch_results:
                                    if not isinstance(result, dict):
                                        with open('error_log.txt', 'a') as f:
                                            f.write(f"Invalid result type in batch_results: {type(result)}\n")
                                        continue
                                    result_addr = result['target']
                                    self.results[result_addr].append(result)
                                    if result_addr not in self.best_matches or result['hamming'] < self.best_matches[result_addr]['hamming']:
                                        self.best_matches[result_addr] = result
                                        if not self.silent_mode or result['hamming'] < 38:
                                            if result['exact_match']:
                                                print(f"{Fore.GREEN}EXACT MATCH FOUND for {result_addr[:8]}...{result_addr[-4:]}")
                                            elif result['hamming'] < 38:
                                                print(f"{Fore.YELLOW}BRUTE-FORCE RANGE for {result_addr[:8]}...{result_addr[-4:]} (Hamming: {result['hamming']})")
                            except Exception as e:
                                with open('error_log.txt', 'a') as f:
                                    f.write(f"Error processing future result: {str(e)}\n")

                    if iteration % 50 == 0 or iteration == iterations-1:
                        self.silent_mode = False
                        self.display_progress()
                        self.save_checkpoint()
                        self.update_report_file()
                    else:
                        self.save_checkpoint()

            self.silent_mode = False
            self.save_checkpoint()
            self.generate_final_report()

            elapsed = time.time() - start_time
            print(f"{Fore.GREEN}Analysis completed in {elapsed:.2f} seconds")
            print(f"{Fore.GREEN}Final report saved to {self.report_file}")
            print(f"{Fore.GREEN}Results saved to breaker_results.txt")

            try:
                files.download(self.report_file)
                print(f"{Fore.GREEN}Report downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download report automatically. Please download it manually.")

            try:
                files.download('best_matches.csv')
                print(f"{Fore.GREEN}Best matches CSV downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download CSV automatically. Please download it manually.")

            try:
                files.download('breaker_results.txt')
                print(f"{Fore.GREEN}Breaker results downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download breaker_results.txt automatically. Please download it manually.")

            try:
                files.download('error_log.txt')
                print(f"{Fore.GREEN}Error log downloaded.")
            except:
                print(f"{Fore.YELLOW}Couldn't download error_log.txt automatically. Please download it manually.")

            return True

        except KeyboardInterrupt:
            print(f"\n{Fore.YELLOW}Analysis interrupted. Saving progress...")
            self.save_checkpoint()
            self.update_report_file()
            return False
        except Exception as e:
            print(f"{Fore.RED}Unexpected error: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Unexpected error in run: {str(e)}\n")
            return False

    def update_report_file(self):
        try:
            with open(self.report_file, 'a') as f:
                f.write(f"\n=== UPDATE: {time.ctime()} ===\n")
                exact_matches = sum(1 for addr in self.target_addresses
                                   if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
                f.write(f"Exact matches found: {exact_matches}/{len(self.target_addresses)}\n")
                if self.best_matches:
                    sorted_matches = sorted(
                        [(addr, match) for addr, match in self.best_matches.items()],
                        key=lambda x: x[1]['hamming']
                    )
                    f.write(f"\nTOP 5 CLOSEST MATCHES:\n")
                    for i, (addr, match) in enumerate(sorted_matches[:5]):
                        if match['exact_match']:
                            f.write(f"#{i+1}: {addr} EXACT MATCH!\n")
                            f.write(f"    Private Key: {match['private_key']}\n")
                            f.write(f"    Entropy Level: {match['entropy']:.2f}/10\n")
                        else:
                            closeness = (160 - match['hamming']) / 160 * 100
                            f.write(f"#{i+1}: {addr} (Hamming: {match['hamming']}, {closeness:.2f}% similar)\n")
                            f.write(f"    Private Key: {match['private_key']}\n")
                            f.write(f"    Entropy Level: {match['entropy']:.2f}/10\n")
                            if match['hamming'] < 38:
                                f.write(f"    **BRUTE-FORCE CANDIDATE**\n")
                            elif match['hamming'] <= 45:
                                f.write(f"    **CLOSE MATCH**\n")
        except Exception as e:
            print(f"{Fore.RED}Error updating report file: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Error updating report file: {str(e)}\n")

    def display_progress(self):
        if self.silent_mode:
            return
        print(f"\n{Fore.CYAN}Progress Update:")
        exact_matches = sum(1 for addr in self.target_addresses
                           if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
        print(f"{Fore.GREEN}Exact matches found: {exact_matches}/{len(self.target_addresses)}")
        if self.best_matches:
            sorted_matches = sorted(
                [(addr, match) for addr, match in self.best_matches.items()],
                key=lambda x: x[1]['hamming']
            )
            print(f"\n{Fore.CYAN}Top 3 Closest Matches:")
            for i, (addr, match) in enumerate(sorted_matches[:3]):
                if match['exact_match']:
                    print(f"{Fore.GREEN}#{i+1}: {addr[:8]}...{addr[-8:]} EXACT MATCH!")
                    print(f"{Fore.GREEN}    Private Key: {match['private_key']}")
                    print(f"{Fore.GREEN}    Entropy Level: {match['entropy']:.2f}/10")
                else:
                    closeness = (160 - match['hamming']) / 160 * 100
                    print(f"{Fore.YELLOW}#{i+1}: {addr[:8]}...{addr[-8:]} (Hamming: {match['hamming']}, {closeness:.2f}% similar)")
                    print(f"{Fore.YELLOW}    Private Key: {match['private_key']}")
                    print(f"{Fore.YELLOW}    Entropy Level: {match['entropy']:.2f}/10")
                    if match['hamming'] < 38:
                        print(f"{Fore.YELLOW}    **BRUTE-FORCE CANDIDATE**")
                    elif match['hamming'] <= 45:
                        print(f"{Fore.YELLOW}    **CLOSE MATCH**")

    def generate_final_report(self):
        try:
            with open(self.report_file, 'a') as f:
                f.write(f"\n\n=========== FINAL KEY BREAKER ANALYSIS REPORT ===========\n")
                f.write(f"Generated at: {time.ctime()}\n\n")
                for addr in self.target_addresses:
                    if addr not in self.best_matches:
                        continue
                    f.write(f"\nAddress: {addr}\n")
                    match = self.best_matches[addr]
                    if match['exact_match']:
                        f.write(f"EXACT MATCH FOUND!\n")
                        f.write(f"Private Key: {match['private_key']}\n")
                        f.write(f"Entropy Level: {match['entropy']:.2f}/10\n")
                    else:
                        closeness = (160 - match['hamming']) / 160 * 100
                        f.write(f"Best Match: (Hamming: {match['hamming']}, {closeness:.2f}% similar)\n")
                        f.write(f"Private Key: {match['private_key']}\n")
                        f.write(f"Entropy Level: {match['entropy']:.2f}/10\n")
                        if match['hamming'] < 38:
                            f.write(f"**BRUTE-FORCE CANDIDATE**\n")
                        elif match['hamming'] <= 45:
                            f.write(f"**CLOSE MATCH**\n")
            with open("best_matches.csv", "w") as csv_file:
                csv_file.write("Address,Private Key,Hamming Distance,Similarity %,Entropy Level,Exact Match\n")
                for addr, match in self.best_matches.items():
                    similarity = (160 - match['hamming']) / 160 * 100 if not match['exact_match'] else 100
                    csv_file.write(f"{addr},{match['private_key']},{match['hamming']},{similarity:.2f},{match['entropy']:.2f},{match['exact_match']}\n")
            print(f"{Fore.GREEN}Best matches also saved to best_matches.csv")
        except Exception as e:
            print(f"{Fore.RED}Error generating final report: {str(e)}")
            with open('error_log.txt', 'a') as f:
                f.write(f"Error generating final report: {str(e)}\n")
            print(f"\n{Fore.CYAN}===== KEY BREAKER ANALYSIS SUMMARY =====")
            exact_matches = sum(1 for addr in self.target_addresses
                               if addr in self.best_matches and self.best_matches[addr].get('exact_match', False))
            print(f"{Fore.GREEN}Exact matches found: {exact_matches}/{len(self.target_addresses)}")
            close_matches = [(addr, match) for addr, match in self.best_matches.items()
                            if not match.get('exact_match', False) and match['hamming'] < 38]
            if close_matches:
                print(f"{Fore.YELLOW}Found {len(close_matches)} brute-force candidates (Hamming < 38)")
                print(f"{Fore.YELLOW}Check the report file for details")

def main():
    print(f"""
{Fore.CYAN}╔══════════════════════════════════════════════════════════════════╗
{Fore.CYAN}║                                                                  ║
{Fore.CYAN}║  {Fore.GREEN}BITCOIN PRIVATE KEY BREAKER{Fore.CYAN}                                 ║
{Fore.CYAN}║  {Fore.YELLOW}Liberal Exploration Around Close Keys from close_keys.txt{Fore.CYAN} ║
{Fore.CYAN}╚══════════════════════════════════════════════════════════════════╝
    """)
    breaker = FastKeyBreaker()
    iterations = 5000
    batch_factor = 2.0
    print(f"{Fore.CYAN}Running with {iterations} iterations and batch size factor {batch_factor}")
    success = breaker.run(iterations=iterations, batch_factor=batch_factor)
    if not success:
        print(f"{Fore.RED}Analysis failed. Check error_log.txt for details.")

if __name__ == "__main__":
    main()


╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║  BITCOIN PRIVATE KEY BREAKER                                 ║
║  Liberal Exploration Around Close Keys from close_keys.txt ║
╚══════════════════════════════════════════════════════════════════╝
    
Running with 5000 iterations and batch size factor 2.0
Loaded checkpoint with 999 addresses.
Starting key breaker analysis for 999 addresses
Using 10 close keys from close_keys.txt (Hamming 39–43)
Using 8 CPU cores for maximum speed
Press Ctrl+C to pause analysis and save progress
Report will be saved to breaker_report.txt

Processing key 1/10 for 1Mp5TX53... (Hamming: 39, Significant: 18f9d8)

Iteration 1/5000 for 1Mp5TX53...

Progress Update:
Exact matches found: 0/999

Top 3 Closest Matches:
#1: 1Mp5TX53...7i2gah13 (Hamming: 39, 75.62% similar)
    Private Key: 000000000000000000000000000000000000000000000000000000000018f9d8
    Entropy Level: 0.00

SystemExit: 0

In [ ]:
#!/usr/bin/env python3
# Simple Bitcoin Keyspace Analyzer - No Batches
# Just keeps all 999 addresses in memory and tracks new bests

# Install required packages
!pip install -q base58 colorama tqdm ecdsa pycryptodome

import hashlib
import base58
import os
import time
from colorama import Fore, init
from tqdm import tqdm
from google.colab import files
import binascii
import ecdsa
from Crypto.Hash import RIPEMD160

# Initialize colorama
init(autoreset=True)

# Configuration
RANGE_START = 40_000_000  # Start of key range to analyze
RANGE_END = 71_000_000    # End of key range to analyze
STEP_SIZE = 100_000       # Progress updates every this many keys
OUTPUT_DIR = "keyspace_analysis"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# File paths
LOG_FILE = f"{OUTPUT_DIR}/analysis_log.txt"
NEW_BESTS_FILE = f"{OUTPUT_DIR}/new_bests.csv"
FINAL_REPORT = f"{OUTPUT_DIR}/final_report.txt"

# Function to generate Bitcoin address from private key integer
def generate_address(private_key_int):
    try:
        # Format private key as hex
        private_key_hex = format(private_key_int, '064x')

        # Convert to ECDSA key
        sk = ecdsa.SigningKey.from_string(bytes.fromhex(private_key_hex), curve=ecdsa.SECP256k1)
        vk = sk.get_verifying_key()

        # Create public key and hash
        public_key = b'\x04' + vk.to_string()
        sha256_hash = hashlib.sha256(public_key).digest()

        # Apply RIPEMD160
        ripemd160_hasher = RIPEMD160.new()
        ripemd160_hasher.update(sha256_hash)
        hash160 = ripemd160_hasher.digest()

        # Create address with version and checksum
        versioned_hash = b'\x00' + hash160
        checksum = hashlib.sha256(hashlib.sha256(versioned_hash).digest()).digest()[:4]
        address_bytes = versioned_hash + checksum

        # Encode as Base58
        address = base58.b58encode(address_bytes).decode('utf-8')
        return address, hash160
    except Exception as e:
        print(f"Error generating address for key {private_key_int}: {str(e)}")
        return None, None

# Function to calculate Hamming distance between two binary objects
def hamming_distance(hash1, hash2):
    try:
        # Ensure both hashes are of same length
        if not hash1 or not hash2 or len(hash1) != len(hash2):
            return 160  # Maximum possible Hamming distance for 160-bit hash

        # Calculate Hamming distance bit by bit
        hamming = 0
        for a, b in zip(hash1, hash2):
            xor_byte = a ^ b
            # Count bits in XOR result
            while xor_byte:
                hamming += xor_byte & 1
                xor_byte >>= 1
        return hamming
    except Exception as e:
        print(f"Error calculating Hamming distance: {str(e)}")
        return 160

# Simple function to load Bitcoin addresses
def load_addresses():
    print(f"{Fore.CYAN}Loading target Bitcoin addresses...")

    try:
        # Check if addresses.txt exists
        if os.path.exists("addresses.txt"):
            with open("addresses.txt", 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]
        else:
            # Prompt for upload
            print(f"{Fore.CYAN}Upload a text file with Bitcoin addresses (one per line)")
            uploaded = files.upload()

            if not uploaded:
                print(f"{Fore.RED}No file uploaded. Exiting.")
                return [], {}

            filename = list(uploaded.keys())[0]
            with open(filename, 'r') as f:
                addresses = [line.strip() for line in f if line.strip()]

        # Validate addresses and extract hash160 values
        valid_addresses = []
        hash160s = {}

        for addr in addresses:
            if not addr.startswith('1'):  # Only P2PKH addresses
                continue

            try:
                decoded = base58.b58decode(addr)
                if len(decoded) != 25:
                    continue

                hash160 = decoded[1:-4]  # Extract RIPEMD160 hash
                valid_addresses.append(addr)
                hash160s[addr] = hash160
            except Exception as e:
                print(f"Error decoding address {addr}: {str(e)}")
                continue

        if not valid_addresses:
            print(f"{Fore.RED}No valid P2PKH addresses found.")
            return [], {}

        print(f"{Fore.GREEN}Loaded {len(valid_addresses)} valid addresses for comparison.")
        return valid_addresses, hash160s

    except Exception as e:
        print(f"{Fore.RED}Error loading addresses: {str(e)}")
        return [], {}

# Function to initialize result tracking
def initialize_tracking(addresses):
    # Initialize tracking dictionaries - all start with max distance
    best_distances = {addr: 160 for addr in addresses}
    best_keys = {addr: None for addr in addresses}
    best_addresses = {addr: None for addr in addresses}

    # Initialize log file
    with open(LOG_FILE, 'w') as f:
        f.write("=== BITCOIN KEYSPACE ANALYSIS - SIMPLE VERSION ===\n")
        f.write(f"Started: {time.ctime()}\n")
        f.write(f"Analyzing key range: {RANGE_START:,} to {RANGE_END:,}\n")
        f.write(f"Target addresses: {len(addresses)}\n\n")
        f.write("Target addresses:\n")
        for addr in addresses:
            f.write(f"  {addr}\n")
        f.write("\n")

    # Initialize new bests file
    with open(NEW_BESTS_FILE, 'w') as f:
        f.write("Timestamp,TargetAddress,PrivateKey,GeneratedAddress,HammingDistance,PreviousBest,Improvement,Range\n")

    return best_distances, best_keys, best_addresses

# Main analysis function
def analyze_range(start_key, end_key, target_addresses, target_hash160s):
    # Initialize tracking
    best_distances, best_keys, best_addresses = initialize_tracking(target_addresses)

    # Stats tracking
    improvements_count = 0
    start_time = time.time()

    # Create progress bar
    total_keys = end_key - start_key
    progress_bar = tqdm(total=total_keys, desc="Analyzing keys")

    # Process each key in range
    current_key = start_key
    last_update = start_time

    while current_key < end_key:
        # Generate address from private key
        address, hash160 = generate_address(current_key)

        if address and hash160:
            # Compare against all target addresses
            for target_addr, target_hash160 in target_hash160s.items():
                # Calculate Hamming distance
                dist = hamming_distance(hash160, target_hash160)

                # Check if this is a new best
                if dist < best_distances[target_addr]:
                    prev_dist = best_distances[target_addr]
                    improvement = prev_dist - dist

                    # Update best results
                    best_distances[target_addr] = dist
                    best_keys[target_addr] = current_key
                    best_addresses[target_addr] = address

                    # Get current range in millions
                    current_range = f"{current_key // 1_000_000}M"

                    # Print new best immediately
                    print(f"\n{Fore.GREEN}🔥 NEW BEST! Target: {target_addr}")
                    print(f"{Fore.GREEN}    Previous best: {prev_dist} bits")
                    print(f"{Fore.GREEN}    New best: {dist} bits (improved by {improvement} bits)")
                    print(f"{Fore.GREEN}    Key: {current_key} (hex: {format(current_key, '064x')})")
                    print(f"{Fore.GREEN}    Generated address: {address}")
                    print(f"{Fore.GREEN}    Range: {current_range}")

                    # Log the new best
                    timestamp = time.strftime("%Y-%m-%d %H:%M:%S")

                    with open(NEW_BESTS_FILE, 'a') as f:
                        f.write(f"{timestamp},{target_addr},{format(current_key, '064x')}," +
                                f"{address},{dist},{prev_dist},{improvement},{current_range}\n")

                    with open(LOG_FILE, 'a') as f:
                        f.write(f"[{timestamp}] NEW BEST for {target_addr}\n")
                        f.write(f"  Previous best: {prev_dist} bits\n")
                        f.write(f"  New best: {dist} bits (improved by {improvement} bits)\n")
                        f.write(f"  Key: {current_key} (hex: {format(current_key, '064x')})\n")
                        f.write(f"  Generated address: {address}\n")
                        f.write(f"  Range: {current_range}\n\n")

                    improvements_count += 1

        # Update progress every STEP_SIZE keys
        if current_key % STEP_SIZE == 0:
            # Update progress bar
            progress_bar.update(STEP_SIZE)

            # Print stats every minute
            current_time = time.time()
            if current_time - last_update >= 60:
                keys_processed = current_key - start_key
                elapsed = current_time - start_time
                keys_per_second = keys_processed / elapsed if elapsed > 0 else 0
                under_40_count = sum(1 for dist in best_distances.values() if dist < 40)

                range_str = f"{(current_key // 1_000_000)}M"
                print(f"\n{Fore.CYAN}Progress Update - Range: {range_str}")
                print(f"{Fore.CYAN}  Keys processed: {keys_processed:,} ({keys_per_second:.2f} keys/sec)")
                print(f"{Fore.CYAN}  Improvements found: {improvements_count}")
                print(f"{Fore.CYAN}  Addresses with Hamming < 40: {under_40_count}/{len(target_addresses)}")

                # Estimate remaining time
                remaining_keys = end_key - current_key
                est_remaining_time = remaining_keys / keys_per_second if keys_per_second > 0 else 0
                print(f"{Fore.CYAN}  Estimated remaining time: {est_remaining_time:.2f} seconds")

                last_update = current_time

        # Move to next key
        current_key += 1

    # Close progress bar
    progress_bar.close()

    # Generate final report
    generate_report(target_addresses, best_distances, best_keys, best_addresses)

    # Show final stats
    elapsed = time.time() - start_time
    print(f"\n{Fore.GREEN}Analysis completed in {elapsed:.2f} seconds")
    print(f"{Fore.GREEN}Found {improvements_count} improvements")
    under_40_count = sum(1 for dist in best_distances.values() if dist < 40)
    print(f"{Fore.GREEN}Addresses with Hamming < 40: {under_40_count}/{len(target_addresses)}")

    # Return results
    return best_distances, best_keys, best_addresses

# Function to generate final report
def generate_report(target_addresses, best_distances, best_keys, best_addresses):
    try:
        with open(FINAL_REPORT, 'w') as f:
            f.write("=== BITCOIN KEYSPACE ANALYSIS - FINAL REPORT ===\n")
            f.write(f"Generated: {time.ctime()}\n")
            f.write(f"Key range analyzed: {RANGE_START:,} to {RANGE_END:,}\n")
            f.write(f"Target addresses: {len(target_addresses)}\n\n")

            # Overall statistics
            min_hamming = min(best_distances.values()) if best_distances else 160
            under_40_count = sum(1 for dist in best_distances.values() if dist < 40)

            f.write(f"Overall minimum Hamming distance: {min_hamming} bits\n")
            f.write(f"Addresses with Hamming distance < 40: {under_40_count}/{len(target_addresses)}\n\n")

            # Sort addresses by Hamming distance
            sorted_addresses = sorted(
                [(addr, best_distances[addr]) for addr in target_addresses],
                key=lambda x: x[1]
            )

            # Best results section
            f.write("Best results:\n\n")

            for addr, dist in sorted_addresses:
                if dist < 160:  # Only include addresses with actual matches
                    f.write(f"Target address: {addr}\n")
                    f.write(f"Best Hamming distance: {dist} bits\n")
                    if best_keys[addr] is not None:
                        f.write(f"Private key: {best_keys[addr]} (hex: {format(best_keys[addr], '064x')})\n")
                        f.write(f"Generated address: {best_addresses[addr]}\n")
                    f.write("\n")

            # Summary by Hamming distance ranges
            f.write("Summary by Hamming distance ranges:\n\n")
            ranges = [(0, 30), (30, 35), (35, 40), (40, 45), (45, 50), (50, 160)]

            for low, high in ranges:
                count = sum(1 for dist in best_distances.values() if low <= dist < high)
                f.write(f"  {low}-{high} bits: {count} addresses\n")

        print(f"{Fore.GREEN}Final report saved to {FINAL_REPORT}")

    except Exception as e:
        print(f"{Fore.RED}Error generating report: {str(e)}")

# Main function
def main():
    print(f"""
{Fore.CYAN}╔══════════════════════════════════════════════════════════════════╗
{Fore.CYAN}║                                                                  ║
{Fore.CYAN}║  {Fore.GREEN}SIMPLE BITCOIN KEYSPACE ANALYZER{Fore.CYAN}                            ║
{Fore.CYAN}║  {Fore.YELLOW}Analyzing Range 40M-71M (No Batching){Fore.CYAN}                      ║
{Fore.CYAN}╚══════════════════════════════════════════════════════════════════╝
    """)

    # Load addresses
    target_addresses, target_hash160s = load_addresses()

    if not target_addresses:
        print(f"{Fore.RED}No valid addresses loaded. Exiting.")
        return

    print(f"{Fore.GREEN}Starting simple keyspace analysis - showing only new best matches")
    print(f"{Fore.GREEN}Range: {RANGE_START:,} to {RANGE_END:,}")
    print(f"{Fore.GREEN}Tracking {len(target_addresses)} addresses in memory")

    # Run analysis
    best_distances, best_keys, best_addresses = analyze_range(
        RANGE_START, RANGE_END, target_addresses, target_hash160s
    )

    # Provide download links
    print(f"\n{Fore.CYAN}Download results:")
    try:
        files.download(FINAL_REPORT)
        files.download(NEW_BESTS_FILE)
        files.download(LOG_FILE)
    except Exception as e:
        print(f"{Fore.YELLOW}Error providing downloads: {str(e)}")
        print(f"{Fore.YELLOW}Files are saved in the {OUTPUT_DIR} directory")

if __name__ == "__main__":
    main()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.6/150.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 38.0 MB/s eta 0:00:00

╔══════════════════════════════════════════════════════════════════╗
║                                                                  ║
║  SIMPLE BITCOIN KEYSPACE ANALYZER                            ║
║  Analyzing Range 40M-71M (No Batching)                      ║
╚══════════════════════════════════════════════════════════════════╝
    
Loading target Bitcoin addresses...
Upload a text file with Bitcoin addresses (one per line)
